# Exploratory data analysis of track 1

Black-box understanding of the English data before any model is trained: the five facets the professor asked for, each computed by a tested function in `dico_nli.eda`, summarised by gold label and drawn as one figure per facet.

**What this notebook does**

1. Loads train and dev, and records how much of dev is already visible in train.
2. Computes the five facets on train: statistical, readability, lexical, semantic, linguistic.
3. Summarises each facet by gold label, saves the tables, and draws one paper-ready figure per facet through `dico_nli.plotting`.
4. Records the run.

Facets are computed on **train** only. Dev is the course test set, so it appears in the split comparison tables and nowhere else.

**Where the results go**

| Output | Path |
|---|---|
| Summary tables | `experiments/2026-09-29-eda-track1/tables/*.csv` |
| Figures | `experiments/2026-09-29-eda-track1/figures/*.pdf`, with `.png` previews |
| Run record | `experiments/2026-09-29-eda-track1/config.json` |
| Written findings | `experiments/2026-09-29-eda-track1/findings.md` |
| Per-instance features | `artifacts/eda-track1/*.csv` (ignored by git) |

**How to run**

```bash
PYTHONIOENCODING=utf-8 uv run jupyter execute notebooks/assignment-1/02_eda_track1.ipynb
```

Needs the task repository cloned at `datasets/dico-nli/`. The semantic facet downloads `sentence-transformers/all-MiniLM-L6-v2` (about 90 MB) on first use.

## 1. Setup

Imports, the run parameters and the paper figure theme. `ORDER` fixes the label order used in every table and figure; `SHORT` gives the axis labels.

In [ ]:
from datetime import date

import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

from dico_nli import eda
from dico_nli.data import LABELS, REPO_ROOT, load_reference
from dico_nli.experiments import experiment_dir, write_config
from dico_nli.plotting import (INK, INK_SECONDARY, LABEL_COLORS, SEQUENTIAL, bar_labels, figure,
                               label_palette, paper_theme, save_figure)

TRACK = 1
RUN_DATE = date(2026, 9, 29)
OVERWRITE = True
SEMANTIC_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SPACY_MODEL = "en_core_web_sm"

ORDER = list(LABELS)
SHORT = {"EQUIVALENCE": "Equiv.", "FORWARD_ENTAILMENT": "Forward",
         "BACKWARD_ENTAILMENT": "Backward", "NEGATIVE_OTHER": "Neg./other"}
paper_theme()

## 2. Helpers

Four small functions used by every facet below.

| Helper | Does |
|---|---|
| `by_label` | aggregates a per-instance frame by gold label, in canonical order |
| `share_table` | row-normalised crosstab of the label against a categorical column |
| `y_labels` | puts the four labels on the y axis of a horizontal bar chart |
| `show` | saves a figure as PDF and PNG through `save_figure`, then displays the PNG |

In [ ]:
def by_label(frame, **aggregations):
    return frame.groupby("label", sort=False).agg(**aggregations).reindex(ORDER)


def share_table(frame, column):
    return pd.crosstab(frame["label"], frame[column], normalize="index").reindex(ORDER)


def y_labels(ax, names=SHORT):
    ax.set_yticks(range(len(ORDER)))
    ax.set_yticklabels([names[label] for label in ORDER])
    ax.invert_yaxis()
    ax.grid(False, axis="y")
    ax.grid(True, axis="x")


def show(fig, stem):
    pdf_path, png_path = save_figure(fig, figures / stem)
    display(Image(filename=str(png_path)))
    return pdf_path

## 3. Data

Train and dev from the reference files, and the run folder. `OVERWRITE = True` replaces an earlier run of the same date; `findings.md` in that folder is written by hand and survives a rerun.

In [ ]:
train = load_reference(TRACK, "train")
dev = load_reference(TRACK, "dev")
out = experiment_dir(f"eda-track{TRACK}", day=RUN_DATE, overwrite=OVERWRITE)
tables, figures = out / "tables", out / "figures"
tables.mkdir(exist_ok=True)
per_instance = REPO_ROOT / "artifacts" / f"eda-track{TRACK}"
per_instance.mkdir(parents=True, exist_ok=True)
print(f"train {len(train)}  dev {len(dev)}  ->  {out.relative_to(REPO_ROOT)}")

## 4. Statistical facet

Label balance per split, the pair structure, overlap between the splits, and phrase lengths.

Every reversible source pair appears in both orders, and NEGATIVE_OTHER instances have no flipped twin. The length question matters most: in a FORWARD_ENTAILMENT pair the premise is usually the longer, more specific phrase ("a red sports car" entails "a car"), so *which side is longer* is a shallow cue for direction that any model can pick up.

In [ ]:
dist = pd.concat([eda.label_distribution(train).assign(split="train"),
                  eda.label_distribution(dev).assign(split="dev")])
dist.to_csv(tables / "label_distribution.csv", index=False)
structure = pd.concat([eda.pair_structure(train).assign(split="train"),
                       eda.pair_structure(dev).assign(split="dev")])
structure.to_csv(tables / "pair_structure.csv", index=False)
overlap = eda.split_overlap(train, dev)
overlap.to_csv(tables / "train_dev_overlap.csv", index=False)

display(dist.pivot(index="label", columns="split", values=["count", "share"]).reindex(ORDER).round(3))
display(structure.pivot(index="quantity", columns="split", values="value"))
display(overlap.set_index("quantity"))

**Figure 1.** Label distribution in train and dev. Forward and backward counts are equal by construction.

In [ ]:
fig, axes = figure(columns=2, ncols=2, height=1.9)
for ax, split in zip(axes, ("train", "dev")):
    part = dist[dist.split == split].set_index("label").reindex(ORDER)
    bars = ax.barh(range(len(ORDER)), part["count"], color=label_palette(), height=0.62)
    bar_labels(ax, bars, labels=[f"{c:,}  ({s:.0%})" for c, s in zip(part["count"], part["share"])])
    y_labels(ax)
    ax.set_title(f"{split.capitalize()} ({int(part['count'].sum()):,} instances)")
    ax.set_xlim(0, part["count"].max() * 1.35)
    ax.set_xlabel("instances")
axes[1].set_yticklabels([])
show(fig, "f1_label_distribution");

In [ ]:
lengths = eda.length_features(train)
lengths.to_csv(per_instance / "length_features.csv", index=False)
length_by_label = by_label(lengths, words_premise=("words1", "mean"), words_hypothesis=("words2", "mean"),
                           word_diff=("word_diff", "mean"), chars_premise=("chars1", "mean"),
                           chars_hypothesis=("chars2", "mean"), max_words=("words1", "max"))
length_by_label.to_csv(tables / "length_by_label.csv")
longer_side = share_table(lengths, "longer_side").reindex(columns=["text2", "equal", "text1"]).fillna(0)
longer_side.to_csv(tables / "longer_side_by_label.csv")
display(length_by_label.round(2))
display(longer_side.rename(columns={"text2": "hypothesis longer", "equal": "same length", "text1": "premise longer"}).round(3))

**Figure 2.** Which phrase is longer, by gold label: the shallow cue for direction.

In [ ]:
colours = {"text2": SEQUENTIAL[1], "equal": SEQUENTIAL[3], "text1": SEQUENTIAL[5]}
names = {"text2": "hypothesis longer", "equal": "same length", "text1": "premise longer"}
fig, ax = figure(columns=1, height=1.9)
left = np.zeros(len(ORDER))
for column in longer_side.columns:
    values = longer_side[column].to_numpy()
    ax.barh(range(len(ORDER)), values, left=left, color=colours[column], height=0.62,
            label=names[column], edgecolor="white", linewidth=0.8)
    for y, (x0, v) in enumerate(zip(left, values)):
        if v >= 0.08:
            ax.text(x0 + v / 2, y, f"{v:.0%}", ha="center", va="center", fontsize=6.5,
                    color="white" if column == "text1" else INK)
    left += values
y_labels(ax)
ax.set_xlim(0, 1)
ax.set_xticks([0, 0.5, 1])
ax.set_xticklabels(["0", "50 %", "100 %"])
ax.set_title("Which phrase is longer, by gold label")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3, handlelength=1.0, columnspacing=1.0)
show(fig, "f2_length_cue");

## 5. Readability facet

Syllables per word, characters per word and Flesch reading ease per phrase, from `textstat`. Flesch-style formulas assume sentences; on two- and three-word phrases they mostly track syllable density, so the table reports them once with that caveat and the figure shows the two simpler measures.

In [ ]:
read = eda.readability_features(train)
read.to_csv(per_instance / "readability_features.csv", index=False)
readability_by_label = by_label(
    read, syllables_per_word_premise=("syllables_per_word1", "mean"), syllables_per_word_hypothesis=("syllables_per_word2", "mean"),
    word_length_premise=("mean_word_length1", "mean"), word_length_hypothesis=("mean_word_length2", "mean"),
    flesch_premise=("flesch1", "median"), flesch_hypothesis=("flesch2", "median"))
readability_by_label.to_csv(tables / "readability_by_label.csv")
display(readability_by_label.round(2))

**Figure 3.** Syllables and characters per word, by gold label.

In [ ]:
pooled = pd.concat([
    read[["label", "syllables_per_word1", "mean_word_length1"]].rename(columns=lambda c: c.rstrip("1")),
    read[["label", "syllables_per_word2", "mean_word_length2"]].rename(columns=lambda c: c.rstrip("2")),
])
summary = by_label(pooled, syllables=("syllables_per_word", "mean"), word_length=("mean_word_length", "mean"))
fig, axes = figure(columns=2, ncols=2, height=1.7)
for ax, column, title, fmt in zip(axes, ("syllables", "word_length"),
                                  ("Syllables per word", "Characters per word"), ("{:.2f}", "{:.1f}")):
    bars = ax.barh(range(len(ORDER)), summary[column], color=label_palette(), height=0.62)
    bar_labels(ax, bars, fmt=fmt)
    y_labels(ax)
    ax.set_title(title)
    ax.set_xlim(0, summary[column].max() * 1.25)
axes[1].set_yticklabels([])
show(fig, "f3_readability");

## 6. Lexical facet

Word-set overlap (Jaccard), containment in each direction, shared first and last token, and the most frequent tokens per label. Containment is the directional cousin of the length cue: when every hypothesis word appears in the premise, the premise is the more specific phrase.

In [ ]:
lex = eda.lexical_features(train)
lex.to_csv(per_instance / "lexical_features.csv", index=False)
lexical_by_label = by_label(
    lex, jaccard=("jaccard", "mean"), jaccard_median=("jaccard", "median"), shared_tokens=("shared_tokens", "mean"),
    premise_in_hypothesis=("containment_1_in_2", "mean"), hypothesis_in_premise=("containment_2_in_1", "mean"),
    shared_first=("shared_first", "mean"), shared_last=("shared_last", "mean"))
lexical_by_label.to_csv(tables / "lexical_by_label.csv")
top_tokens = eda.top_tokens_by_label(train, n=15)
top_tokens.to_csv(tables / "top_tokens_by_label.csv", index=False)
display(lexical_by_label.round(3))
display(top_tokens.groupby("label", sort=False).head(8).pivot_table(index=top_tokens.groupby("label").cumcount(), columns="label", values="token", aggfunc="first").reindex(columns=ORDER))

**Figure 4.** Left: Jaccard overlap by label. Right: how often one phrase's words are all contained in the other's, by direction.

In [ ]:
fig, axes = figure(columns=2, ncols=2, height=1.9)
ax = axes[0]
sns.boxplot(data=lex, y="label", x="jaccard", order=ORDER, hue="label", palette=LABEL_COLORS, legend=False,
            width=0.62, fliersize=1.5, linewidth=0.7, ax=ax)
ax.set_yticks(range(len(ORDER)))
ax.set_yticklabels([SHORT[label] for label in ORDER])
ax.set_ylabel("")
ax.set_xlabel("Jaccard overlap of word sets")
ax.set_title("Lexical overlap")
ax.grid(False, axis="y")
ax.grid(True, axis="x")

ax = axes[1]
full = by_label(lex.assign(h_in_p=lex["containment_2_in_1"] >= 1.0, p_in_h=lex["containment_1_in_2"] >= 1.0),
                h_in_p=("h_in_p", "mean"), p_in_h=("p_in_h", "mean"))
y = np.arange(len(ORDER))
ax.barh(y - 0.17, full["p_in_h"], height=0.32, color=SEQUENTIAL[3], label="premise words all in hypothesis")
ax.barh(y + 0.17, full["h_in_p"], height=0.32, color=SEQUENTIAL[5], label="hypothesis words all in premise")
for yy, (a, b) in enumerate(zip(full["p_in_h"], full["h_in_p"])):
    ax.text(a + 0.02, yy - 0.17, f"{a:.0%}", va="center", fontsize=6.5, color=INK_SECONDARY)
    ax.text(b + 0.02, yy + 0.17, f"{b:.0%}", va="center", fontsize=6.5, color=INK_SECONDARY)
y_labels(ax)
ax.set_yticklabels([])
ax.set_xlim(0, 1.15)
ax.set_xticks([0, 0.5, 1])
ax.set_xticklabels(["0", "50 %", "100 %"])
ax.set_title("Full containment, by direction")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=1, handlelength=1.0)
show(fig, "f4_lexical");

## 7. Semantic facet

Cosine similarity between the two phrases under a sentence encoder (`all-MiniLM-L6-v2`). Cosine is symmetric, so an original and its flipped twin get the same value: this signal can separate related pairs from NEGATIVE_OTHER but carries no information about direction. The figure shows FORWARD and BACKWARD as one curve; they are identical by construction.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(SEMANTIC_MODEL, device="cpu")
sem = eda.semantic_similarity(train, encoder=lambda texts: model.encode(texts, batch_size=64, normalize_embeddings=True))
sem.to_csv(per_instance / "semantic_similarity.csv", index=False)
equivalence_median = sem.loc[sem.label == "EQUIVALENCE", "cosine"].median()
semantic_by_label = by_label(sem.assign(above_equiv_median=sem["cosine"] >= equivalence_median),
                             cosine=("cosine", "mean"), cosine_median=("cosine", "median"), cosine_std=("cosine", "std"),
                             share_above_equivalence_median=("above_equiv_median", "mean"))
semantic_by_label.to_csv(tables / "semantic_by_label.csv")
display(semantic_by_label.round(3))

**Figure 5.** Distribution of cosine similarity by label; the directional labels share one curve.

In [ ]:
directional = sem[sem.label.isin(["FORWARD_ENTAILMENT", "BACKWARD_ENTAILMENT"])]["cosine"]
series = [
    ("EQUIVALENCE", sem.loc[sem.label == "EQUIVALENCE", "cosine"], "Equiv."),
    ("FORWARD_ENTAILMENT", directional, "Forward = Backward (symmetric)"),
    ("NEGATIVE_OTHER", sem.loc[sem.label == "NEGATIVE_OTHER", "cosine"], "Neg./other"),
]
fig, ax = figure(columns=1, height=2.0)
for label, values, name in series:
    sns.kdeplot(values, ax=ax, color=LABEL_COLORS[label], fill=True, alpha=0.10, linewidth=1.2,
                clip=(-0.2, 1.0), bw_adjust=0.9, label=f"{name}, median {values.median():.2f}")
ax.set_xlim(-0.1, 1.0)
ax.set_xlabel("cosine similarity, MiniLM-L6 embeddings")
ax.set_ylabel("density")
ax.set_title("Semantic similarity of the two phrases")
ax.grid(False)
ax.legend(loc="upper left", handlelength=1.0)
show(fig, "f5_semantic");

## 8. Linguistic facet

spaCy (`en_core_web_sm`) part-of-speech and dependency patterns: is the head word the same on both sides, is the POS pattern identical, does either phrase contain a preposition (the pilot paper's main error source), and how many adjectives and numerals each side carries.

In [ ]:
import spacy

ling = eda.linguistic_features(train, nlp=spacy.load(SPACY_MODEL))
ling.to_csv(per_instance / "linguistic_features.csv", index=False)
ling = ling.assign(any_prep=ling["has_prep1"] | ling["has_prep2"],
                   adj_premise_minus_hypothesis=ling["n_adj1"] - ling["n_adj2"],
                   num_any=(ling["n_num1"] + ling["n_num2"]) > 0)
linguistic_by_label = by_label(
    ling, same_head=("same_head", "mean"), same_pos_pattern=("same_pos_pattern", "mean"),
    any_preposition=("any_prep", "mean"), adj_premise_minus_hypothesis=("adj_premise_minus_hypothesis", "mean"),
    any_numeral=("num_any", "mean"))
linguistic_by_label.to_csv(tables / "linguistic_by_label.csv")
pos_patterns = (ling.groupby("label")["pos1"].value_counts().groupby(level=0).head(5).rename("count")
                .reset_index().rename(columns={"pos1": "premise_pos_pattern"}))
pos_patterns.to_csv(tables / "top_pos_patterns_by_label.csv", index=False)
display(linguistic_by_label.round(3))
display(pos_patterns)

**Figure 6.** Same head word, same POS pattern and presence of a preposition, by gold label.

In [ ]:
fig, axes = figure(columns=2, ncols=3, height=1.7)
for ax, column, title in zip(axes, ("same_head", "same_pos_pattern", "any_preposition"),
                             ("Same head word", "Same POS pattern", "Preposition in a phrase")):
    bars = ax.barh(range(len(ORDER)), linguistic_by_label[column], color=label_palette(), height=0.62)
    bar_labels(ax, bars, labels=[f"{v:.0%}" for v in linguistic_by_label[column]])
    y_labels(ax)
    ax.set_title(title)
    ax.set_xlim(0, 1.2)
    ax.set_xticks([0, 0.5, 1])
    ax.set_xticklabels(["0", "50 %", "100 %"])
for ax in axes[1:]:
    ax.set_yticklabels([])
show(fig, "f6_linguistic");

## 9. Run record

`config.json` records the data commit, this repository's commit and the command. The list below is every file the run wrote.

In [ ]:
write_config(
    out, seed=0, track=TRACK,
    command="PYTHONIOENCODING=utf-8 uv run jupyter execute notebooks/assignment-1/02_eda_track1.ipynb",
    extra={"note": "deterministic; seed unused", "semantic_model": SEMANTIC_MODEL, "spacy_model": SPACY_MODEL,
           "facets_on": "train", "dev_used_for": ["label_distribution", "pair_structure", "train_dev_overlap"],
           "train_size": len(train), "dev_size": len(dev)},
)
sorted(p.relative_to(out).as_posix() for p in out.rglob("*") if p.is_file())